In [20]:
from langgraph.graph import START, END, StateGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Annotated



In [21]:
llm = ChatGoogleGenerativeAI(model = 'gemini-3.1-flash-lite')

In [23]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [32]:
def generate_joke(state: JokeState): 

    prompt = f"Generate a joke about {state['topic']}"
    response = llm.invoke(prompt).text

    return {'joke': response}


In [33]:
def generate_explanation(state: JokeState):

    prompt = f"Explain the joke: {state['joke']}"
    response = llm.invoke(prompt).text

    return {'explanation': response}


In [34]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer = checkpointer)

In [35]:
config1 = {'configurable': {'thread_id': "1"}}

initial_state = {'topic': 'programming'}

workflow.invoke(initial_state, config = config1)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'topic': 'programming',
 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.',
 'explanation': 'The joke relies on a **pun** based on two different meanings of the word "bug."\n\n1.  **In nature:** Light sources (like porch lights or lamps) attract insects, which are commonly called "bugs."\n2.  **In programming:** A "bug" is an error, flaw, or glitch in a computer program that causes it to behave unexpectedly.\n\n**The humor:**\nProgrammers spend their days trying to identify and fix "bugs" (errors) in their code. By saying "light attracts bugs," the joke playfully suggests that if a programmer uses "light mode" (a bright white screen background), it will literally draw in computer errors—just like a porch light draws in moths and mosquitoes.'}

In [36]:
config2 = {'configurable': {'thread_id': "2"}}

initial_state = {'topic': 'pizza'}

workflow.invoke(initial_state, config = config2)

{'topic': 'pizza',
 'joke': 'Why does pizza travel so well?\n\nBecause it’s always delivered in a box!',
 'explanation': 'The humor in this joke comes from a **play on words** (a pun) involving the word "box."\n\nHere is the breakdown:\n\n1.  **The setup:** "Why does pizza travel so well?" implies a question about the pizza\'s physical durability or how easy it is to transport.\n2.  **The expectation:** When people hear "travel," they often think of a person or an object moving from place to place.\n3.  **The twist:** The word **"box"** is being used in two different ways:\n    *   **The literal meaning:** Pizzas are physically carried in a cardboard box, which keeps them secure and easy to carry.\n    *   **The idiomatic meaning:** There is a common phrase, "thinking outside the box," which refers to being creative. By saying it’s "delivered in a box," the joke plays on the idea that the pizza is literally inside a box while it’s "traveling" to your house.\n\nEssentially, it’s a "dad 

In [37]:
workflow.get_state(config = config1)

StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explanation': 'The joke relies on a **pun** based on two different meanings of the word "bug."\n\n1.  **In nature:** Light sources (like porch lights or lamps) attract insects, which are commonly called "bugs."\n2.  **In programming:** A "bug" is an error, flaw, or glitch in a computer program that causes it to behave unexpectedly.\n\n**The humor:**\nProgrammers spend their days trying to identify and fix "bugs" (errors) in their code. By saying "light attracts bugs," the joke playfully suggests that if a programmer uses "light mode" (a bright white screen background), it will literally draw in computer errors—just like a porch light draws in moths and mosquitoes.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bde6a-946b-661c-8002-256112b9a43e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at

In [38]:
workflow.get_state(config = config2)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why does pizza travel so well?\n\nBecause it’s always delivered in a box!', 'explanation': 'The humor in this joke comes from a **play on words** (a pun) involving the word "box."\n\nHere is the breakdown:\n\n1.  **The setup:** "Why does pizza travel so well?" implies a question about the pizza\'s physical durability or how easy it is to transport.\n2.  **The expectation:** When people hear "travel," they often think of a person or an object moving from place to place.\n3.  **The twist:** The word **"box"** is being used in two different ways:\n    *   **The literal meaning:** Pizzas are physically carried in a cardboard box, which keeps them secure and easy to carry.\n    *   **The idiomatic meaning:** There is a common phrase, "thinking outside the box," which refers to being creative. By saying it’s "delivered in a box," the joke plays on the idea that the pizza is literally inside a box while it’s "traveling" to your house.\n\nEssent

In [39]:
list(workflow.get_state_history(config = config1))

[StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explanation': 'The joke relies on a **pun** based on two different meanings of the word "bug."\n\n1.  **In nature:** Light sources (like porch lights or lamps) attract insects, which are commonly called "bugs."\n2.  **In programming:** A "bug" is an error, flaw, or glitch in a computer program that causes it to behave unexpectedly.\n\n**The humor:**\nProgrammers spend their days trying to identify and fix "bugs" (errors) in their code. By saying "light attracts bugs," the joke playfully suggests that if a programmer uses "light mode" (a bright white screen background), it will literally draw in computer errors—just like a porch light draws in moths and mosquitoes.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bde6a-946b-661c-8002-256112b9a43e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_a

In [40]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [48]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [49]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(10)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [50]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [51]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [52]:
graph.get_state({"configurable": {"thread_id": 'thread-1'}})

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bde8d-b954-65b4-8001-889f04de6bf0'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-10-01T22:38:43.925120+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bde8d-b950-6bf8-8000-643f7e3c2ad6'}}, tasks=(PregelTask(id='3bb3e724-3d1b-0730-fa46-3f39ac91fb49', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [53]:
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}
